In [8]:
"""LACI with SAOCP-style absolute-residual score (four-grid experiment).

The original LACI alpha update, local-error weights, rolling train/calibration
split, LSTM quantile regressors, initial training and subsequent training
cadence, data preprocessing, and metrics helper are retained.

CHANGED from signed CQR LACI:
    point_t = (q_lower_t + q_upper_t) / 2
    score_t = |y_t - point_t|
    radius_t = quantile(absolute calibration residuals, adaptive level)
    C_t = [point_t - radius_t, point_t + radius_t]

This is a score/interval-family ablation of LACI, not the original CQR-LACI.
Requires the existing ../Datasets CSV files and Helper_Scripts modules.
Run: python run_laci_midpoint_absolute_score_4grids.py
Test: python run_laci_midpoint_absolute_score_4grids.py --self-test
"""
from __future__ import annotations

import argparse
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
SEQ_LEN = 48
EPOCHS = 200
LR = 0.001
BATCH_SIZE = 32
ALPHA = 0.10
TINIT = 5000
SPLIT = 0.50
GAMMA = 0.001
TRAIN_EVERY = 6
DOWNSAMPLE_STRIDE = 4
LOCAL_ERROR_WINDOW = 24
RECENCY_DECAY = 0.85
MAGNITUDE_EPS = 1e-4  # matches scoring_function_v1 in supplied LACI code
LOG_EVERY = 200
SAVE_TRAJECTORIES = True

# Same three datasets enabled in user's supplied four-grid script.
# Add '../Datasets/50Hertz.csv' if desired.
DATASET_PATHS = [
    '../Datasets/50Hertz.csv',
    '../Datasets/Amprion.csv',
    '../Datasets/TenneTTSO.csv',
    '../Datasets/TransnetBW.csv',
]
OUTPUT_CSV = './results_LACI_midpoint_absolute_4grids.csv'
TRAJECTORY_DIR = './trajectories/LSTM_LACI_midpoint_absolute'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_sequences(series: np.ndarray, seq_len: int):
    windows = np.lib.stride_tricks.sliding_window_view(series, seq_len + 1)
    X = np.ascontiguousarray(windows[:, :-1], dtype=np.float32)
    y = np.asarray(windows[:, -1], dtype=np.float32)
    return X, y


def load_dataset(filepath: str):
    # Preserve flatten-every-column-except-first and stride=4 preprocessing.
    df = pd.read_csv(filepath)
    raw = df.iloc[:, 1:].to_numpy().flatten().astype(float)[::DOWNSAMPLE_STRIDE]
    if len(raw) <= SEQ_LEN + TINIT:
        raise ValueError(f'{filepath}: too few observations after downsampling')
    if not np.isfinite(raw).all():
        raise ValueError(f'{filepath}: nonfinite observations in series')
    scaler = StandardScaler()
    scaler.fit(raw[:TINIT + SEQ_LEN].reshape(-1, 1))
    series = scaler.transform(raw.reshape(-1, 1)).reshape(-1)
    X, y = build_sequences(series, SEQ_LEN)
    return X, y, scaler


def quantile_loss(y_true, y_pred, quantile):
    diff = y_true - y_pred
    return torch.mean(torch.maximum(quantile * diff, (quantile - 1.0) * diff))


def make_models():
    from Helper_Scripts.models_TLMR import LSTMQuantileReg
    lo_model = LSTMQuantileReg().to(DEVICE)
    hi_model = LSTMQuantileReg().to(DEVICE)
    lo_optimizer = torch.optim.Adam(lo_model.parameters(), lr=LR)
    hi_optimizer = torch.optim.Adam(hi_model.parameters(), lr=LR)
    return (lo_model, hi_model), (lo_optimizer, hi_optimizer)


def fit_models(models, optimizers, X_train, y_train, epochs):
    if epochs == 0:
        return
    ds = TensorDataset(torch.as_tensor(X_train, dtype=torch.float32),
                       torch.as_tensor(y_train[:, None], dtype=torch.float32))
    loader = DataLoader(ds, batch_size=min(BATCH_SIZE, len(ds)), shuffle=True)
    for model, opt, quantile in zip(models, optimizers,
                                    [ALPHA / 2.0, 1.0 - ALPHA / 2.0]):
        model.train()
        for _ in range(epochs):
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                loss = quantile_loss(yb, model(xb), quantile)
                opt.zero_grad()
                loss.backward()
                opt.step()


def predict_midpoint(models, X, batch_size=4096):
    """Average the two quantile model outputs, as done for SAOCP."""
    outputs = []
    for model in models:
        model.eval()
        blocks = []
        with torch.no_grad():
            for start in range(0, len(X), batch_size):
                x = torch.as_tensor(X[start:start + batch_size],
                                    dtype=torch.float32, device=DEVICE)
                blocks.append(model(x).detach().cpu().numpy().reshape(-1))
        outputs.append(np.concatenate(blocks).astype(np.float64))
    return 0.5 * (outputs[0] + outputs[1])


def absolute_score(y, point):
    return np.abs(np.asarray(y, dtype=np.float64) -
                  np.asarray(point, dtype=np.float64))


def calibrated_radius(cal_scores, alpha_t):
    """Preserve user's adaptive quantile convention, now for |y - midpoint|.

    This matches the original LACI code's numpy quantile interpolation; it
    is not the finite-sample conformal order-statistic convention.
    """
    level = np.clip((1.0 - alpha_t) * (1.0 + 1.0 / len(cal_scores)), 0, 1)
    return float(np.quantile(cal_scores, level))


def weighted_laci_error(error_history, magnitude_history):
    """Same weighted-error formula as scoring_function_v1 in the input."""
    if not error_history:
        return 0.0
    n = len(error_history)
    exponent = np.arange(n - 1, -1, -1, dtype=np.float64)
    recency = RECENCY_DECAY ** exponent
    weights = recency * (np.asarray(magnitude_history, dtype=np.float64)
                         + MAGNITUDE_EPS)
    weights /= weights.sum()
    return float(np.dot(weights, np.asarray(error_history, dtype=np.float64)))


def run_laci(X, y, scaler, name):
    from Helper_Scripts.calculate_metrics import (
        summarize_all_metrics, WINDOW as LOCAL_WINDOW, CWC_ETA
    )
    set_seed(SEED)
    if len(y) <= TINIT:
        raise ValueError(f'{name}: insufficient test observations')
    models, opts = make_models()
    n_train = int(round(TINIT * SPLIT))
    if not 0 < n_train < TINIT:
        raise ValueError('Invalid train/calibration split')
    alpha_t = ALPHA
    error_history = []
    magnitude_history = []
    covered_history = []
    rows = []
    t0 = time.monotonic()

    for t in range(TINIT, len(y)):
        # Original LACI: rolling TINIT-length window with oldest 50% for
        # training and newest 50% for calibration. Preserve this order.
        begin = t - TINIT
        tr = slice(begin, begin + n_train)
        cal = slice(begin + n_train, t)

        if t == TINIT:
            epochs = EPOCHS
        elif (t - TINIT) % TRAIN_EVERY == 0:
            epochs = 1
        else:
            epochs = 0
        fit_models(models, opts, X[tr], y[tr], epochs)

        # Recompute calibration scores using the current LSTM, same as LACI.
        point_cal = predict_midpoint(models, X[cal])
        cal_scores = absolute_score(y[cal], point_cal)
        alpha_used = float(np.clip(alpha_t, 1e-6, 1.0 - 1e-6))
        radius = calibrated_radius(cal_scores, alpha_used)

        # Predict interval BEFORE observing current y[t].
        point_t = float(predict_midpoint(models, X[t:t+1])[0])
        lower = point_t - radius
        upper = point_t + radius
        y_t = float(y[t])
        score_t = float(abs(y_t - point_t))
        err = float(not (lower <= y_t <= upper))

        # Original LACI local feedback and recency/magnitude weighting.
        covered_history.append(1.0 - err)
        if len(covered_history) >= LOCAL_ERROR_WINDOW:
            local_picp = float(np.mean(covered_history[-LOCAL_ERROR_WINDOW:]))
            magnitude = max((1.0 - ALPHA) - local_picp, 0.0)
        else:
            magnitude = 0.0
        error_history.append(err)
        magnitude_history.append(magnitude)
        weighted_err = weighted_laci_error(error_history, magnitude_history)
        alpha_t = float(np.clip(alpha_used + GAMMA * (ALPHA - weighted_err),
                                1e-6, 1.0 - 1e-6))

        rows.append((t - TINIT, t, alpha_t, alpha_used, err, y_t,
                     lower, upper, point_t, radius, score_t, weighted_err,
                     magnitude, float(np.mean(cal_scores))))
        if LOG_EVERY and (t - TINIT + 1) % LOG_EVERY == 0:
            running = 1.0 - np.mean([r[4] for r in rows])
            print(f'[{name}/LACI] {t-TINIT+1}/{len(y)-TINIT} '
                  f'PICP={running:.4f} alpha={alpha_t:.4f} '
                  f'radius={radius:.4f}', flush=True)

    columns = ['step', 'dataset_index', 'alphaTrajectory', 'alpha_used',
               'errSeq', 'y_true', 'y_l_pred', 'y_u_pred', 'point_pred',
               'radius', 'score', 'weighted_err', 'local_error',
               'mean_cal_score']
    trajectory = pd.DataFrame(rows, columns=columns)
    original_scale = float(scaler.scale_[0])
    original_offset = float(scaler.mean_[0])
    for col in ['y_true', 'y_l_pred', 'y_u_pred', 'point_pred']:
        trajectory[col] = trajectory[col] * original_scale + original_offset
    for col in ['radius', 'score', 'mean_cal_score']:
        trajectory[col] *= original_scale
    trajectory['covered'] = 1.0 - trajectory['errSeq']
    lp = trajectory['covered'].rolling(LOCAL_WINDOW, min_periods=LOCAL_WINDOW).mean()
    trajectory['LPICP'] = lp
    trajectory['LE'] = ((1.0 - ALPHA) - lp).clip(lower=0)
    trajectory['local_overcoverage'] = (lp - (1.0 - ALPHA)).clip(lower=0)
    trajectory['local_abs_coverage_error'] = (lp - (1.0 - ALPHA)).abs()
    trajectory['local_MPIW'] = (trajectory['y_u_pred'] - trajectory['y_l_pred']).rolling(
        LOCAL_WINDOW, min_periods=LOCAL_WINDOW).mean()

    elapsed = time.monotonic() - t0
    metrics = summarize_all_metrics(
        trajectory['y_true'].to_numpy(),
        trajectory['y_l_pred'].to_numpy(),
        trajectory['y_u_pred'].to_numpy(),
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA, time_taken=elapsed,
    )
    metrics['dataset'] = name + '.csv'
    metrics['method'] = 'LACI_midpoint_absolute'
    metrics['time_sec'] = elapsed
    metrics['mean_radius'] = float(trajectory['radius'].mean())
    metrics['median_radius'] = float(trajectory['radius'].median())
    metrics['mean_alpha'] = float(trajectory['alpha_used'].mean())
    metrics['mean_weighted_err'] = float(trajectory['weighted_err'].mean())

    if SAVE_TRAJECTORIES:
        outdir = Path(TRAJECTORY_DIR)
        outdir.mkdir(parents=True, exist_ok=True)
        trajectory.to_csv(outdir / f'{name}__LACI_midpoint_absolute.csv', index=False)

    to_show = ['PICP', 'MPIW', 'LPICE', 'LPIA', 'LMPIW', 'time_sec']
    print(f'[{name}/LACI_midpoint_absolute] ' + ' | '.join(
        f'{k}={float(metrics[k]):.4f}' for k in to_show if k in metrics
    ), flush=True)
    return metrics


def self_test():
    # New nonconformity and interval inclusion are exactly equivalent.
    x = np.array([3.0, 5.0, 10.0])
    pred = np.array([4.0, 5.0, 7.0])
    assert np.allclose(absolute_score(x, pred), [1.0, 0.0, 3.0])
    for point in [0.0, 3.0]:
        for target in [-2.0, 0.0, 1.0, 5.0]:
            for rad in [0.0, 1.0, 4.0]:
                assert (point - rad <= target <= point + rad) == (abs(target-point) <= rad)
    assert weighted_laci_error([0, 1], [0, 0]) > 0.5
    assert np.isclose(weighted_laci_error([0, 1], [0, 0]), 1.0/(1.0+0.85))
    assert calibrated_radius(np.array([0.0, 1.0, 2.0, 3.0]), 0.1) >= 0
    print('Self-test passed: absolute score, symmetric interval, LACI weights, radius.')


def main(paths):
    print('Device:', DEVICE)
    all_metrics = []
    for path in paths:
        if not Path(path).is_file():
            raise FileNotFoundError(f'Missing dataset: {path}')
        name = Path(path).stem
        print(f'\nRunning {name} ...', flush=True)
        X, y, scaler = load_dataset(path)
        result = run_laci(X, y, scaler, name)
        all_metrics.append(result)
        pd.DataFrame(all_metrics).to_csv(OUTPUT_CSV, index=False)
    result_df = pd.DataFrame(all_metrics)
    keys = ['dataset', 'method', 'PICP', 'MPIW', 'MLPICP', 'LPICE',
            'LPIA', 'LMPIW', 'LCWC', 'STD_LE', 'n_local_windows', 'time_sec']
    keys = [k for k in keys if k in result_df.columns]
    result_df = result_df[keys + [k for k in result_df.columns if k not in keys]]
    result_df.to_csv(OUTPUT_CSV, index=False)
    print('\nResults saved:', OUTPUT_CSV)
    print(result_df.to_string(index=False))


if __name__ == '__main__':
   
        main(DATASET_PATHS)


Device: cuda

Running 50Hertz ...
[50Hertz/LACI] 200/4480 PICP=0.9300 alpha=0.0824 radius=0.2850
[50Hertz/LACI] 400/4480 PICP=0.9500 alpha=0.0948 radius=0.3072
[50Hertz/LACI] 600/4480 PICP=0.9567 alpha=0.1076 radius=0.2828
[50Hertz/LACI] 800/4480 PICP=0.9537 alpha=0.1044 radius=0.3017
[50Hertz/LACI] 1000/4480 PICP=0.9580 alpha=0.1183 radius=0.2743
[50Hertz/LACI] 1200/4480 PICP=0.9625 alpha=0.1324 radius=0.2430
[50Hertz/LACI] 1400/4480 PICP=0.9643 alpha=0.1463 radius=0.2087
[50Hertz/LACI] 1600/4480 PICP=0.9619 alpha=0.1543 radius=0.1854
[50Hertz/LACI] 1800/4480 PICP=0.9600 alpha=0.1598 radius=0.1675
[50Hertz/LACI] 2000/4480 PICP=0.9555 alpha=0.1519 radius=0.1592
[50Hertz/LACI] 2200/4480 PICP=0.9577 alpha=0.1650 radius=0.1428
[50Hertz/LACI] 2400/4480 PICP=0.9546 alpha=0.1634 radius=0.1313
[50Hertz/LACI] 2600/4480 PICP=0.9404 alpha=0.1287 radius=0.1498
[50Hertz/LACI] 2800/4480 PICP=0.9343 alpha=0.1136 radius=0.1593
[50Hertz/LACI] 3000/4480 PICP=0.9353 alpha=0.1183 radius=0.1459
[50Hertz/L

KeyboardInterrupt: 

In [3]:
"""LACI with SAOCP-style absolute-residual score (hydropower experiment).

The original LACI alpha update, local-error weights, rolling train/calibration
split, LSTM quantile regressors, initial training and subsequent training
cadence, data preprocessing, and metrics helper are retained.

CHANGED from signed CQR LACI:
    point_t = (q_lower_t + q_upper_t) / 2
    score_t = |y_t - point_t|
    radius_t = quantile(absolute calibration residuals, adaptive level)
    C_t = [point_t - radius_t, point_t + radius_t]

This is a score/interval-family ablation of LACI, not the original CQR-LACI.
The rolling training/calibration protocol matches the earlier LACI script,
not the fixed initial SAOCP calibration history. No SAOCP experts are used.
Requires the existing ../Datasets CSV files and Helper_Scripts modules.
Run: python run_laci_midpoint_absolute_hydropower.py
Test: python run_laci_midpoint_absolute_hydropower.py --self-test
"""
from __future__ import annotations

import argparse
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
SEQ_LEN = 48
EPOCHS = 200
LR = 0.001
BATCH_SIZE = 32
ALPHA = 0.10
TINIT = 200
SPLIT = 0.50
GAMMA = 0.001
TRAIN_EVERY = 6
LOCAL_ERROR_WINDOW = 24
RECENCY_DECAY = 0.85
MAGNITUDE_EPS = 1e-4  # matches scoring_function_v1 in supplied LACI code
LOG_EVERY = 200
SAVE_TRAJECTORIES = True

# Exactly the hydropower dataset used in the supplied SAOCP script.
DATASET_PATHS = [
    '../Datasets/hydropower_dataset.csv',
]
OUTPUT_CSV = './results_LACI_midpoint_absolute_hydropower.csv'
TRAJECTORY_DIR = './trajectories/LSTM_LACI_midpoint_absolute_hydropower'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_sequences(series: np.ndarray, seq_len: int):
    windows = np.lib.stride_tricks.sliding_window_view(series, seq_len + 1)
    X = np.ascontiguousarray(windows[:, :-1], dtype=np.float32)
    y = np.asarray(windows[:, -1], dtype=np.float32)
    return X, y


def load_dataset(filepath: str):
    """Hydropower preprocessing identical to the supplied SAOCP hydro run.

    total = Actual Day Peak (MW) + Actual Eve Peak (MW)
    Drop rows whose total is NaN or zero, then form 48-step sequences.
    The removal of records may compress the original timeline.
    """
    df = pd.read_csv(filepath)
    required = ['Actual Day Peak (MW)', 'Actual Eve Peak (MW)']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f'{filepath}: missing columns {missing}')
    day = pd.to_numeric(df[required[0]], errors='coerce')
    evening = pd.to_numeric(df[required[1]], errors='coerce')
    total = day + evening
    raw = total.loc[total.notna() & (total != 0)].to_numpy(dtype=np.float64)
    if not np.isfinite(raw).all():
        raise ValueError(f'{filepath}: filtered data contains non-finite values')
    if len(raw) <= SEQ_LEN + TINIT:
        raise ValueError(f'{filepath}: too few retained observations for TINIT={TINIT}')
    scaler = StandardScaler()
    scaler.fit(raw[:TINIT + SEQ_LEN].reshape(-1, 1))
    series = scaler.transform(raw.reshape(-1, 1)).reshape(-1)
    X, y = build_sequences(series, SEQ_LEN)
    return X, y, scaler


def quantile_loss(y_true, y_pred, quantile):
    diff = y_true - y_pred
    return torch.mean(torch.maximum(quantile * diff, (quantile - 1.0) * diff))


def make_models():
    from Helper_Scripts.models_TLMR import LSTMQuantileReg
    lo_model = LSTMQuantileReg().to(DEVICE)
    hi_model = LSTMQuantileReg().to(DEVICE)
    lo_optimizer = torch.optim.Adam(lo_model.parameters(), lr=LR)
    hi_optimizer = torch.optim.Adam(hi_model.parameters(), lr=LR)
    return (lo_model, hi_model), (lo_optimizer, hi_optimizer)


def fit_models(models, optimizers, X_train, y_train, epochs):
    if epochs == 0:
        return
    ds = TensorDataset(torch.as_tensor(X_train, dtype=torch.float32),
                       torch.as_tensor(y_train[:, None], dtype=torch.float32))
    loader = DataLoader(ds, batch_size=min(BATCH_SIZE, len(ds)), shuffle=True)
    for model, opt, quantile in zip(models, optimizers,
                                    [ALPHA / 2.0, 1.0 - ALPHA / 2.0]):
        model.train()
        for _ in range(epochs):
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                loss = quantile_loss(yb, model(xb), quantile)
                opt.zero_grad()
                loss.backward()
                opt.step()


def predict_midpoint(models, X, batch_size=4096):
    """Average the two quantile model outputs, as done for SAOCP."""
    outputs = []
    for model in models:
        model.eval()
        blocks = []
        with torch.no_grad():
            for start in range(0, len(X), batch_size):
                x = torch.as_tensor(X[start:start + batch_size],
                                    dtype=torch.float32, device=DEVICE)
                blocks.append(model(x).detach().cpu().numpy().reshape(-1))
        outputs.append(np.concatenate(blocks).astype(np.float64))
    return 0.5 * (outputs[0] + outputs[1])


def absolute_score(y, point):
    return np.abs(np.asarray(y, dtype=np.float64) -
                  np.asarray(point, dtype=np.float64))


def calibrated_radius(cal_scores, alpha_t):
    """Preserve user's adaptive quantile convention, now for |y - midpoint|.

    This matches the original LACI code's numpy quantile interpolation; it
    is not the finite-sample conformal order-statistic convention.
    """
    level = np.clip((1.0 - alpha_t) * (1.0 + 1.0 / len(cal_scores)), 0, 1)
    return float(np.quantile(cal_scores, level))


def weighted_laci_error(error_history, magnitude_history):
    """Same weighted-error formula as scoring_function_v1 in the input."""
    if not error_history:
        return 0.0
    n = len(error_history)
    exponent = np.arange(n - 1, -1, -1, dtype=np.float64)
    recency = RECENCY_DECAY ** exponent
    weights = recency * (np.asarray(magnitude_history, dtype=np.float64)
                         + MAGNITUDE_EPS)
    weights /= weights.sum()
    return float(np.dot(weights, np.asarray(error_history, dtype=np.float64)))


def run_laci(X, y, scaler, name):
    from Helper_Scripts.calculate_metrics import (
        summarize_all_metrics, WINDOW as LOCAL_WINDOW, CWC_ETA
    )
    set_seed(SEED)
    if len(y) <= TINIT:
        raise ValueError(f'{name}: insufficient test observations')
    models, opts = make_models()
    n_train = int(round(TINIT * SPLIT))
    if not 0 < n_train < TINIT:
        raise ValueError('Invalid train/calibration split')
    alpha_t = ALPHA
    error_history = []
    magnitude_history = []
    covered_history = []
    rows = []
    t0 = time.monotonic()

    for t in range(TINIT, len(y)):
        # Original LACI: rolling TINIT-length window with oldest 50% for
        # training and newest 50% for calibration. Preserve this order.
        begin = t - TINIT
        tr = slice(begin, begin + n_train)
        cal = slice(begin + n_train, t)

        if t == TINIT:
            epochs = EPOCHS
        elif (t - TINIT) % TRAIN_EVERY == 0:
            epochs = 1
        else:
            epochs = 0
        fit_models(models, opts, X[tr], y[tr], epochs)

        # Recompute calibration scores using the current LSTM, same as LACI.
        point_cal = predict_midpoint(models, X[cal])
        cal_scores = absolute_score(y[cal], point_cal)
        alpha_used = float(np.clip(alpha_t, 1e-6, 1.0 - 1e-6))
        radius = calibrated_radius(cal_scores, alpha_used)

        # Predict interval BEFORE observing current y[t].
        point_t = float(predict_midpoint(models, X[t:t+1])[0])
        lower = point_t - radius
        upper = point_t + radius
        y_t = float(y[t])
        score_t = float(abs(y_t - point_t))
        err = float(not (lower <= y_t <= upper))

        # Original LACI local feedback and recency/magnitude weighting.
        covered_history.append(1.0 - err)
        if len(covered_history) >= LOCAL_ERROR_WINDOW:
            local_picp = float(np.mean(covered_history[-LOCAL_ERROR_WINDOW:]))
            magnitude = max((1.0 - ALPHA) - local_picp, 0.0)
        else:
            magnitude = 0.0
        error_history.append(err)
        magnitude_history.append(magnitude)
        weighted_err = weighted_laci_error(error_history, magnitude_history)
        alpha_t = float(np.clip(alpha_used + GAMMA * (ALPHA - weighted_err),
                                1e-6, 1.0 - 1e-6))

        rows.append((t - TINIT, t, alpha_t, alpha_used, err, y_t,
                     lower, upper, point_t, radius, score_t, weighted_err,
                     magnitude, float(np.mean(cal_scores))))
        if LOG_EVERY and (t - TINIT + 1) % LOG_EVERY == 0:
            running = 1.0 - np.mean([r[4] for r in rows])
            print(f'[{name}/LACI] {t-TINIT+1}/{len(y)-TINIT} '
                  f'PICP={running:.4f} alpha={alpha_t:.4f} '
                  f'radius={radius:.4f}', flush=True)

    columns = ['step', 'dataset_index', 'alphaTrajectory', 'alpha_used',
               'errSeq', 'y_true', 'y_l_pred', 'y_u_pred', 'point_pred',
               'radius', 'score', 'weighted_err', 'local_error',
               'mean_cal_score']
    trajectory = pd.DataFrame(rows, columns=columns)
    original_scale = float(scaler.scale_[0])
    original_offset = float(scaler.mean_[0])
    for col in ['y_true', 'y_l_pred', 'y_u_pred', 'point_pred']:
        trajectory[col] = trajectory[col] * original_scale + original_offset
    for col in ['radius', 'score', 'mean_cal_score']:
        trajectory[col] *= original_scale
    trajectory['covered'] = 1.0 - trajectory['errSeq']
    lp = trajectory['covered'].rolling(LOCAL_WINDOW, min_periods=LOCAL_WINDOW).mean()
    trajectory['LPICP'] = lp
    trajectory['LE'] = ((1.0 - ALPHA) - lp).clip(lower=0)
    trajectory['local_overcoverage'] = (lp - (1.0 - ALPHA)).clip(lower=0)
    trajectory['local_abs_coverage_error'] = (lp - (1.0 - ALPHA)).abs()
    trajectory['local_MPIW'] = (trajectory['y_u_pred'] - trajectory['y_l_pred']).rolling(
        LOCAL_WINDOW, min_periods=LOCAL_WINDOW).mean()

    elapsed = time.monotonic() - t0
    metrics = summarize_all_metrics(
        trajectory['y_true'].to_numpy(),
        trajectory['y_l_pred'].to_numpy(),
        trajectory['y_u_pred'].to_numpy(),
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA, time_taken=elapsed,
    )
    metrics['dataset'] = name + '.csv'
    metrics['method'] = 'LACI_midpoint_absolute'
    metrics['time_sec'] = elapsed
    metrics['mean_radius'] = float(trajectory['radius'].mean())
    metrics['median_radius'] = float(trajectory['radius'].median())
    metrics['mean_alpha'] = float(trajectory['alpha_used'].mean())
    metrics['mean_weighted_err'] = float(trajectory['weighted_err'].mean())

    if SAVE_TRAJECTORIES:
        outdir = Path(TRAJECTORY_DIR)
        outdir.mkdir(parents=True, exist_ok=True)
        trajectory.to_csv(outdir / f'{name}__LACI_midpoint_absolute.csv', index=False)

    to_show = ['PICP', 'MPIW', 'LPICE', 'LPIA', 'LMPIW', 'time_sec']
    print(f'[{name}/LACI_midpoint_absolute] ' + ' | '.join(
        f'{k}={float(metrics[k]):.4f}' for k in to_show if k in metrics
    ), flush=True)
    return metrics


def self_test():
    # New nonconformity and interval inclusion are exactly equivalent.
    x = np.array([3.0, 5.0, 10.0])
    pred = np.array([4.0, 5.0, 7.0])
    assert np.allclose(absolute_score(x, pred), [1.0, 0.0, 3.0])
    for point in [0.0, 3.0]:
        for target in [-2.0, 0.0, 1.0, 5.0]:
            for rad in [0.0, 1.0, 4.0]:
                assert (point - rad <= target <= point + rad) == (abs(target-point) <= rad)
    assert weighted_laci_error([0, 1], [0, 0]) > 0.5
    assert np.isclose(weighted_laci_error([0, 1], [0, 0]), 1.0/(1.0+0.85))
    assert calibrated_radius(np.array([0.0, 1.0, 2.0, 3.0]), 0.1) >= 0
    print('Self-test passed: absolute score, symmetric interval, LACI weights, radius.')


def main(paths):
    print('Device:', DEVICE)
    all_metrics = []
    for path in paths:
        if not Path(path).is_file():
            raise FileNotFoundError(f'Missing dataset: {path}')
        name = Path(path).stem
        print(f'\nRunning {name} ...', flush=True)
        X, y, scaler = load_dataset(path)
        result = run_laci(X, y, scaler, name)
        all_metrics.append(result)
        pd.DataFrame(all_metrics).to_csv(OUTPUT_CSV, index=False)
    result_df = pd.DataFrame(all_metrics)
    keys = ['dataset', 'method', 'PICP', 'MPIW', 'MLPICP', 'LPICE',
            'LPIA', 'LMPIW', 'LCWC', 'STD_LE', 'n_local_windows', 'time_sec']
    keys = [k for k in keys if k in result_df.columns]
    result_df = result_df[keys + [k for k in result_df.columns if k not in keys]]
    result_df.to_csv(OUTPUT_CSV, index=False)
    print('\nResults saved:', OUTPUT_CSV)
    print(result_df.to_string(index=False))


if __name__ == '__main__':
    main(DATASET_PATHS)


Device: cuda

Running hydropower_dataset ...
[hydropower_dataset/LACI] 200/1505 PICP=0.8750 alpha=0.0939 radius=1.1198
[hydropower_dataset/LACI] 400/1505 PICP=0.8925 alpha=0.0859 radius=1.4024
[hydropower_dataset/LACI] 600/1505 PICP=0.9200 alpha=0.0981 radius=0.7267
[hydropower_dataset/LACI] 800/1505 PICP=0.8938 alpha=0.0744 radius=2.6101
[hydropower_dataset/LACI] 1000/1505 PICP=0.9020 alpha=0.0782 radius=0.7789
[hydropower_dataset/LACI] 1200/1505 PICP=0.8875 alpha=0.0534 radius=1.7809
[hydropower_dataset/LACI] 1400/1505 PICP=0.8971 alpha=0.0629 radius=1.2391
[hydropower_dataset/LACI_midpoint_absolute] PICP=0.8990 | MPIW=182.9078 | LPICE=0.0507 | LPIA=0.6653 | LMPIW=182.6546 | time_sec=7.4370

Results saved: ./results_LACI_midpoint_absolute_hydropower.csv
               dataset                 method     PICP       MPIW   MLPICP    LPICE     LPIA      LMPIW         LCWC   STD_LE  n_local_windows  time_sec  mean_radius  median_radius  mean_alpha  mean_weighted_err
hydropower_dataset.csv

In [6]:
"""LACI with SAOCP-style absolute-residual score (Location1 and Location2 experiments).

The original LACI alpha update, local-error weights, rolling train/calibration
split, LSTM quantile regressors, initial training and subsequent training
cadence, data preprocessing, and metrics helper are retained.

CHANGED from signed CQR LACI:
    point_t = (q_lower_t + q_upper_t) / 2
    score_t = |y_t - point_t|
    radius_t = quantile(absolute calibration residuals, adaptive level)
    C_t = [point_t - radius_t, point_t + radius_t]

This is a score/interval-family ablation of LACI, not the original CQR-LACI.
The rolling training/calibration protocol matches the earlier LACI script,
not the fixed initial SAOCP calibration history. No SAOCP experts are used.
Requires the existing ../Datasets CSV files and Helper_Scripts modules.
Run: python run_laci_midpoint_absolute_locations.py
Test: python run_laci_midpoint_absolute_locations.py --self-test
"""
from __future__ import annotations

import argparse
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
SEQ_LEN = 48
EPOCHS = 200
LR = 0.001
BATCH_SIZE = 32
ALPHA = 0.10
TINIT = 5000
SPLIT = 0.50
GAMMA = 0.001
TRAIN_EVERY = 6
LOCAL_ERROR_WINDOW = 24
RECENCY_DECAY = 0.85
MAGNITUDE_EPS = 1e-4  # matches scoring_function_v1 in supplied LACI code
LOG_EVERY = 200
SAVE_TRAJECTORIES = True

# Match the Location1/Location2 SAOCP data selection; no downsampling.
DATASET_PATHS = [
    '../Datasets/Location1.csv',
    '../Datasets/Location2.csv',
]
OUTPUT_CSV = './results_LACI_midpoint_absolute_2locations.csv'
TRAJECTORY_DIR = './trajectories/LSTM_LACI_midpoint_absolute_2locations'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_sequences(series: np.ndarray, seq_len: int):
    windows = np.lib.stride_tricks.sliding_window_view(series, seq_len + 1)
    X = np.ascontiguousarray(windows[:, :-1], dtype=np.float32)
    y = np.asarray(windows[:, -1], dtype=np.float32)
    return X, y


def load_dataset(filepath: str):
    """Match the two-location SAOCP loader: first 10,000 rows of Power.

    Fit the standardizer to the available initial prefix only, then form
    48-observation one-step-ahead input/label pairs. No row filtering or
    downsampling is applied.
    """
    path = Path(filepath)
    if path.stem not in {'Location1', 'Location2'}:
        raise ValueError(f'Expected Location1.csv or Location2.csv: {filepath}')
    df = pd.read_csv(path).iloc[:10000]
    if 'Power' not in df.columns:
        raise ValueError(f'{filepath}: missing Power column')
    raw = pd.to_numeric(df['Power'], errors='raise').to_numpy(dtype=np.float64)
    if len(raw) <= SEQ_LEN + TINIT:
        raise ValueError(f'{filepath}: insufficient observations for TINIT={TINIT}')
    if not np.isfinite(raw).all():
        raise ValueError(f'{filepath}: nonfinite Power values')
    scaler = StandardScaler()
    scaler.fit(raw[:TINIT + SEQ_LEN].reshape(-1, 1))
    series = scaler.transform(raw.reshape(-1, 1)).reshape(-1)
    X, y = build_sequences(series, SEQ_LEN)
    return X, y, scaler


def quantile_loss(y_true, y_pred, quantile):
    diff = y_true - y_pred
    return torch.mean(torch.maximum(quantile * diff, (quantile - 1.0) * diff))


def make_models():
    from Helper_Scripts.models_TLMR import LSTMQuantileReg
    lo_model = LSTMQuantileReg().to(DEVICE)
    hi_model = LSTMQuantileReg().to(DEVICE)
    lo_optimizer = torch.optim.Adam(lo_model.parameters(), lr=LR)
    hi_optimizer = torch.optim.Adam(hi_model.parameters(), lr=LR)
    return (lo_model, hi_model), (lo_optimizer, hi_optimizer)


def fit_models(models, optimizers, X_train, y_train, epochs):
    if epochs == 0:
        return
    ds = TensorDataset(torch.as_tensor(X_train, dtype=torch.float32),
                       torch.as_tensor(y_train[:, None], dtype=torch.float32))
    loader = DataLoader(ds, batch_size=min(BATCH_SIZE, len(ds)), shuffle=True)
    for model, opt, quantile in zip(models, optimizers,
                                    [ALPHA / 2.0, 1.0 - ALPHA / 2.0]):
        model.train()
        for _ in range(epochs):
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                loss = quantile_loss(yb, model(xb), quantile)
                opt.zero_grad()
                loss.backward()
                opt.step()


def predict_midpoint(models, X, batch_size=4096):
    """Average the two quantile model outputs, as done for SAOCP."""
    outputs = []
    for model in models:
        model.eval()
        blocks = []
        with torch.no_grad():
            for start in range(0, len(X), batch_size):
                x = torch.as_tensor(X[start:start + batch_size],
                                    dtype=torch.float32, device=DEVICE)
                blocks.append(model(x).detach().cpu().numpy().reshape(-1))
        outputs.append(np.concatenate(blocks).astype(np.float64))
    return 0.5 * (outputs[0] + outputs[1])


def absolute_score(y, point):
    return np.abs(np.asarray(y, dtype=np.float64) -
                  np.asarray(point, dtype=np.float64))


def calibrated_radius(cal_scores, alpha_t):
    """Preserve user's adaptive quantile convention, now for |y - midpoint|.

    This matches the original LACI code's numpy quantile interpolation; it
    is not the finite-sample conformal order-statistic convention.
    """
    level = np.clip((1.0 - alpha_t) * (1.0 + 1.0 / len(cal_scores)), 0, 1)
    return float(np.quantile(cal_scores, level))


def weighted_laci_error(error_history, magnitude_history):
    """Same weighted-error formula as scoring_function_v1 in the input."""
    if not error_history:
        return 0.0
    n = len(error_history)
    exponent = np.arange(n - 1, -1, -1, dtype=np.float64)
    recency = RECENCY_DECAY ** exponent
    weights = recency * (np.asarray(magnitude_history, dtype=np.float64)
                         + MAGNITUDE_EPS)
    weights /= weights.sum()
    return float(np.dot(weights, np.asarray(error_history, dtype=np.float64)))


def run_laci(X, y, scaler, name):
    from Helper_Scripts.calculate_metrics import (
        summarize_all_metrics, WINDOW as LOCAL_WINDOW, CWC_ETA
    )
    set_seed(SEED)
    if len(y) <= TINIT:
        raise ValueError(f'{name}: insufficient test observations')
    models, opts = make_models()
    n_train = int(round(TINIT * SPLIT))
    if not 0 < n_train < TINIT:
        raise ValueError('Invalid train/calibration split')
    alpha_t = ALPHA
    error_history = []
    magnitude_history = []
    covered_history = []
    rows = []
    t0 = time.monotonic()

    for t in range(TINIT, len(y)):
        # Original LACI: rolling TINIT-length window with oldest 50% for
        # training and newest 50% for calibration. Preserve this order.
        begin = t - TINIT
        tr = slice(begin, begin + n_train)
        cal = slice(begin + n_train, t)

        if t == TINIT:
            epochs = EPOCHS
        elif (t - TINIT) % TRAIN_EVERY == 0:
            epochs = 1
        else:
            epochs = 0
        fit_models(models, opts, X[tr], y[tr], epochs)

        # Recompute calibration scores using the current LSTM, same as LACI.
        point_cal = predict_midpoint(models, X[cal])
        cal_scores = absolute_score(y[cal], point_cal)
        alpha_used = float(np.clip(alpha_t, 1e-6, 1.0 - 1e-6))
        radius = calibrated_radius(cal_scores, alpha_used)

        # Predict interval BEFORE observing current y[t].
        point_t = float(predict_midpoint(models, X[t:t+1])[0])
        lower = point_t - radius
        upper = point_t + radius
        y_t = float(y[t])
        score_t = float(abs(y_t - point_t))
        err = float(not (lower <= y_t <= upper))

        # Original LACI local feedback and recency/magnitude weighting.
        covered_history.append(1.0 - err)
        if len(covered_history) >= LOCAL_ERROR_WINDOW:
            local_picp = float(np.mean(covered_history[-LOCAL_ERROR_WINDOW:]))
            magnitude = max((1.0 - ALPHA) - local_picp, 0.0)
        else:
            magnitude = 0.0
        error_history.append(err)
        magnitude_history.append(magnitude)
        weighted_err = weighted_laci_error(error_history, magnitude_history)
        alpha_t = float(np.clip(alpha_used + GAMMA * (ALPHA - weighted_err),
                                1e-6, 1.0 - 1e-6))

        rows.append((t - TINIT, t, alpha_t, alpha_used, err, y_t,
                     lower, upper, point_t, radius, score_t, weighted_err,
                     magnitude, float(np.mean(cal_scores))))
        if LOG_EVERY and (t - TINIT + 1) % LOG_EVERY == 0:
            running = 1.0 - np.mean([r[4] for r in rows])
            print(f'[{name}/LACI] {t-TINIT+1}/{len(y)-TINIT} '
                  f'PICP={running:.4f} alpha={alpha_t:.4f} '
                  f'radius={radius:.4f}', flush=True)

    columns = ['step', 'dataset_index', 'alphaTrajectory', 'alpha_used',
               'errSeq', 'y_true', 'y_l_pred', 'y_u_pred', 'point_pred',
               'radius', 'score', 'weighted_err', 'local_error',
               'mean_cal_score']
    trajectory = pd.DataFrame(rows, columns=columns)
    original_scale = float(scaler.scale_[0])
    original_offset = float(scaler.mean_[0])
    for col in ['y_true', 'y_l_pred', 'y_u_pred', 'point_pred']:
        trajectory[col] = trajectory[col] * original_scale + original_offset
    for col in ['radius', 'score', 'mean_cal_score']:
        trajectory[col] *= original_scale
    trajectory['covered'] = 1.0 - trajectory['errSeq']
    lp = trajectory['covered'].rolling(LOCAL_WINDOW, min_periods=LOCAL_WINDOW).mean()
    trajectory['LPICP'] = lp
    trajectory['LE'] = ((1.0 - ALPHA) - lp).clip(lower=0)
    trajectory['local_overcoverage'] = (lp - (1.0 - ALPHA)).clip(lower=0)
    trajectory['local_abs_coverage_error'] = (lp - (1.0 - ALPHA)).abs()
    trajectory['local_MPIW'] = (trajectory['y_u_pred'] - trajectory['y_l_pred']).rolling(
        LOCAL_WINDOW, min_periods=LOCAL_WINDOW).mean()

    elapsed = time.monotonic() - t0
    metrics = summarize_all_metrics(
        trajectory['y_true'].to_numpy(),
        trajectory['y_l_pred'].to_numpy(),
        trajectory['y_u_pred'].to_numpy(),
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA, time_taken=elapsed,
    )
    metrics['dataset'] = name + '.csv'
    metrics['method'] = 'LACI_midpoint_absolute'
    metrics['time_sec'] = elapsed
    metrics['mean_radius'] = float(trajectory['radius'].mean())
    metrics['median_radius'] = float(trajectory['radius'].median())
    metrics['mean_alpha'] = float(trajectory['alpha_used'].mean())
    metrics['mean_weighted_err'] = float(trajectory['weighted_err'].mean())

    if SAVE_TRAJECTORIES:
        outdir = Path(TRAJECTORY_DIR)
        outdir.mkdir(parents=True, exist_ok=True)
        trajectory.to_csv(outdir / f'{name}__LACI_midpoint_absolute.csv', index=False)

    to_show = ['PICP', 'MPIW', 'LPICE', 'LPIA', 'LMPIW', 'time_sec']
    print(f'[{name}/LACI_midpoint_absolute] ' + ' | '.join(
        f'{k}={float(metrics[k]):.4f}' for k in to_show if k in metrics
    ), flush=True)
    return metrics


def self_test():
    # New nonconformity and interval inclusion are exactly equivalent.
    x = np.array([3.0, 5.0, 10.0])
    pred = np.array([4.0, 5.0, 7.0])
    assert np.allclose(absolute_score(x, pred), [1.0, 0.0, 3.0])
    for point in [0.0, 3.0]:
        for target in [-2.0, 0.0, 1.0, 5.0]:
            for rad in [0.0, 1.0, 4.0]:
                assert (point - rad <= target <= point + rad) == (abs(target-point) <= rad)
    assert weighted_laci_error([0, 1], [0, 0]) > 0.5
    assert np.isclose(weighted_laci_error([0, 1], [0, 0]), 1.0/(1.0+0.85))
    assert calibrated_radius(np.array([0.0, 1.0, 2.0, 3.0]), 0.1) >= 0
    print('Self-test passed: absolute score, symmetric interval, LACI weights, radius.')


def main(paths):
    print('Device:', DEVICE)
    all_metrics = []
    for path in paths:
        if not Path(path).is_file():
            raise FileNotFoundError(f'Missing dataset: {path}')
        name = Path(path).stem
        print(f'\nRunning {name} ...', flush=True)
        X, y, scaler = load_dataset(path)
        result = run_laci(X, y, scaler, name)
        all_metrics.append(result)
        pd.DataFrame(all_metrics).to_csv(OUTPUT_CSV, index=False)
    result_df = pd.DataFrame(all_metrics)
    keys = ['dataset', 'method', 'PICP', 'MPIW', 'MLPICP', 'LPICE',
            'LPIA', 'LMPIW', 'LCWC', 'STD_LE', 'n_local_windows', 'time_sec']
    keys = [k for k in keys if k in result_df.columns]
    result_df = result_df[keys + [k for k in result_df.columns if k not in keys]]
    result_df.to_csv(OUTPUT_CSV, index=False)
    print('\nResults saved:', OUTPUT_CSV)
    print(result_df.to_string(index=False))


if __name__ == '__main__':
    main(DATASET_PATHS)


Device: cuda

Running Location1 ...
[Location1/LACI] 200/4952 PICP=0.9650 alpha=0.1119 radius=0.0521
[Location1/LACI] 400/4952 PICP=0.9425 alpha=0.1141 radius=0.0678
[Location1/LACI] 600/4952 PICP=0.9383 alpha=0.1142 radius=0.0467
[Location1/LACI] 800/4952 PICP=0.9413 alpha=0.1239 radius=0.0434
[Location1/LACI] 1000/4952 PICP=0.9270 alpha=0.0998 radius=0.0559
[Location1/LACI] 1200/4952 PICP=0.9250 alpha=0.1004 radius=0.0546
[Location1/LACI] 1400/4952 PICP=0.9157 alpha=0.0869 radius=0.0572
[Location1/LACI] 1600/4952 PICP=0.9163 alpha=0.0807 radius=0.0651
[Location1/LACI] 1800/4952 PICP=0.9178 alpha=0.0747 radius=0.0522
[Location1/LACI] 2000/4952 PICP=0.9130 alpha=0.0615 radius=0.0611
[Location1/LACI] 2200/4952 PICP=0.9105 alpha=0.0484 radius=0.0809
[Location1/LACI] 2400/4952 PICP=0.9146 alpha=0.0586 radius=0.0684
[Location1/LACI] 2600/4952 PICP=0.9158 alpha=0.0547 radius=0.0746
[Location1/LACI] 2800/4952 PICP=0.9143 alpha=0.0476 radius=0.0834
[Location1/LACI] 3000/4952 PICP=0.9123 alpha

In [7]:

from __future__ import annotations

import argparse
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
SEQ_LEN = 48
EPOCHS = 200
LR = 0.001
BATCH_SIZE = 32
ALPHA = 0.10
TINIT = 4000
SPLIT = 0.50
GAMMA = 0.001
TRAIN_EVERY = 6
LOCAL_ERROR_WINDOW = 24
RECENCY_DECAY = 0.85
MAGNITUDE_EPS = 1e-4  # matches scoring_function_v1 in supplied LACI code
LOG_EVERY = 200
SAVE_TRAJECTORIES = True

# Match the solar SAOCP data selection: first 10,000 rows, nonzero DAILY_YIELD.
DATASET_PATHS = [
    '../Datasets/Plant_1_Generation_Data.csv',
    '../Datasets/Plant_2_Generation_Data.csv',
]
OUTPUT_CSV = './results_LACI_midpoint_absolute_2plants.csv'
TRAJECTORY_DIR = './trajectories/LSTM_LACI_midpoint_absolute_2plants'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_sequences(series: np.ndarray, seq_len: int):
    windows = np.lib.stride_tricks.sliding_window_view(series, seq_len + 1)
    X = np.ascontiguousarray(windows[:, :-1], dtype=np.float32)
    y = np.asarray(windows[:, -1], dtype=np.float32)
    return X, y


def load_dataset(filepath: str):
    """Replicate the Plant1/Plant2 SAOCP preprocessing.

    Take the first 10,000 rows of DAILY_YIELD; drop missing or zero values,
    without downsampling. Fit the scaler on the initial prefix only.
    Filtering compresses the time axis; local windows count retained samples.
    """
    path = Path(filepath)
    if path.stem not in {'Plant_1_Generation_Data', 'Plant_2_Generation_Data'}:
        raise ValueError(f'Expected Plant_1_Generation_Data.csv or Plant_2_Generation_Data.csv: {filepath}')
    df = pd.read_csv(path).iloc[:10000]
    if 'DAILY_YIELD' not in df.columns:
        raise ValueError(f'{filepath}: missing DAILY_YIELD column')
    values = pd.to_numeric(df['DAILY_YIELD'], errors='coerce')
    raw = values.loc[values.notna() & (values != 0)].to_numpy(dtype=np.float64)
    if len(raw) <= SEQ_LEN + TINIT:
        raise ValueError(f'{filepath}: insufficient retained observations for TINIT={TINIT}')
    if not np.isfinite(raw).all():
        raise ValueError(f'{filepath}: nonfinite DAILY_YIELD values')
    scaler = StandardScaler()
    scaler.fit(raw[:TINIT + SEQ_LEN].reshape(-1, 1))
    series = scaler.transform(raw.reshape(-1, 1)).reshape(-1)
    X, y = build_sequences(series, SEQ_LEN)
    return X, y, scaler


def quantile_loss(y_true, y_pred, quantile):
    diff = y_true - y_pred
    return torch.mean(torch.maximum(quantile * diff, (quantile - 1.0) * diff))


def make_models():
    from Helper_Scripts.models_TLMR import LSTMQuantileReg
    lo_model = LSTMQuantileReg().to(DEVICE)
    hi_model = LSTMQuantileReg().to(DEVICE)
    lo_optimizer = torch.optim.Adam(lo_model.parameters(), lr=LR)
    hi_optimizer = torch.optim.Adam(hi_model.parameters(), lr=LR)
    return (lo_model, hi_model), (lo_optimizer, hi_optimizer)


def fit_models(models, optimizers, X_train, y_train, epochs):
    if epochs == 0:
        return
    ds = TensorDataset(torch.as_tensor(X_train, dtype=torch.float32),
                       torch.as_tensor(y_train[:, None], dtype=torch.float32))
    loader = DataLoader(ds, batch_size=min(BATCH_SIZE, len(ds)), shuffle=True)
    for model, opt, quantile in zip(models, optimizers,
                                    [ALPHA / 2.0, 1.0 - ALPHA / 2.0]):
        model.train()
        for _ in range(epochs):
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                loss = quantile_loss(yb, model(xb), quantile)
                opt.zero_grad()
                loss.backward()
                opt.step()


def predict_midpoint(models, X, batch_size=4096):
    """Average the two quantile model outputs, as done for SAOCP."""
    outputs = []
    for model in models:
        model.eval()
        blocks = []
        with torch.no_grad():
            for start in range(0, len(X), batch_size):
                x = torch.as_tensor(X[start:start + batch_size],
                                    dtype=torch.float32, device=DEVICE)
                blocks.append(model(x).detach().cpu().numpy().reshape(-1))
        outputs.append(np.concatenate(blocks).astype(np.float64))
    return 0.5 * (outputs[0] + outputs[1])


def absolute_score(y, point):
    return np.abs(np.asarray(y, dtype=np.float64) -
                  np.asarray(point, dtype=np.float64))


def calibrated_radius(cal_scores, alpha_t):
    """Preserve user's adaptive quantile convention, now for |y - midpoint|.

    This matches the original LACI code's numpy quantile interpolation; it
    is not the finite-sample conformal order-statistic convention.
    """
    level = np.clip((1.0 - alpha_t) * (1.0 + 1.0 / len(cal_scores)), 0, 1)
    return float(np.quantile(cal_scores, level))


def weighted_laci_error(error_history, magnitude_history):
    """Same weighted-error formula as scoring_function_v1 in the input."""
    if not error_history:
        return 0.0
    n = len(error_history)
    exponent = np.arange(n - 1, -1, -1, dtype=np.float64)
    recency = RECENCY_DECAY ** exponent
    weights = recency * (np.asarray(magnitude_history, dtype=np.float64)
                         + MAGNITUDE_EPS)
    weights /= weights.sum()
    return float(np.dot(weights, np.asarray(error_history, dtype=np.float64)))


def run_laci(X, y, scaler, name):
    from Helper_Scripts.calculate_metrics import (
        summarize_all_metrics, WINDOW as LOCAL_WINDOW, CWC_ETA
    )
    set_seed(SEED)
    if len(y) <= TINIT:
        raise ValueError(f'{name}: insufficient test observations')
    models, opts = make_models()
    n_train = int(round(TINIT * SPLIT))
    if not 0 < n_train < TINIT:
        raise ValueError('Invalid train/calibration split')
    alpha_t = ALPHA
    error_history = []
    magnitude_history = []
    covered_history = []
    rows = []
    t0 = time.monotonic()

    for t in range(TINIT, len(y)):
        # Original LACI: rolling TINIT-length window with oldest 50% for
        # training and newest 50% for calibration. Preserve this order.
        begin = t - TINIT
        tr = slice(begin, begin + n_train)
        cal = slice(begin + n_train, t)

        if t == TINIT:
            epochs = EPOCHS
        elif (t - TINIT) % TRAIN_EVERY == 0:
            epochs = 1
        else:
            epochs = 0
        fit_models(models, opts, X[tr], y[tr], epochs)

        # Recompute calibration scores using the current LSTM, same as LACI.
        point_cal = predict_midpoint(models, X[cal])
        cal_scores = absolute_score(y[cal], point_cal)
        alpha_used = float(np.clip(alpha_t, 1e-6, 1.0 - 1e-6))
        radius = calibrated_radius(cal_scores, alpha_used)

        # Predict interval BEFORE observing current y[t].
        point_t = float(predict_midpoint(models, X[t:t+1])[0])
        lower = point_t - radius
        upper = point_t + radius
        y_t = float(y[t])
        score_t = float(abs(y_t - point_t))
        err = float(not (lower <= y_t <= upper))

        # Original LACI local feedback and recency/magnitude weighting.
        covered_history.append(1.0 - err)
        if len(covered_history) >= LOCAL_ERROR_WINDOW:
            local_picp = float(np.mean(covered_history[-LOCAL_ERROR_WINDOW:]))
            magnitude = max((1.0 - ALPHA) - local_picp, 0.0)
        else:
            magnitude = 0.0
        error_history.append(err)
        magnitude_history.append(magnitude)
        weighted_err = weighted_laci_error(error_history, magnitude_history)
        alpha_t = float(np.clip(alpha_used + GAMMA * (ALPHA - weighted_err),
                                1e-6, 1.0 - 1e-6))

        rows.append((t - TINIT, t, alpha_t, alpha_used, err, y_t,
                     lower, upper, point_t, radius, score_t, weighted_err,
                     magnitude, float(np.mean(cal_scores))))
        if LOG_EVERY and (t - TINIT + 1) % LOG_EVERY == 0:
            running = 1.0 - np.mean([r[4] for r in rows])
            print(f'[{name}/LACI] {t-TINIT+1}/{len(y)-TINIT} '
                  f'PICP={running:.4f} alpha={alpha_t:.4f} '
                  f'radius={radius:.4f}', flush=True)

    columns = ['step', 'dataset_index', 'alphaTrajectory', 'alpha_used',
               'errSeq', 'y_true', 'y_l_pred', 'y_u_pred', 'point_pred',
               'radius', 'score', 'weighted_err', 'local_error',
               'mean_cal_score']
    trajectory = pd.DataFrame(rows, columns=columns)
    original_scale = float(scaler.scale_[0])
    original_offset = float(scaler.mean_[0])
    for col in ['y_true', 'y_l_pred', 'y_u_pred', 'point_pred']:
        trajectory[col] = trajectory[col] * original_scale + original_offset
    for col in ['radius', 'score', 'mean_cal_score']:
        trajectory[col] *= original_scale
    trajectory['covered'] = 1.0 - trajectory['errSeq']
    lp = trajectory['covered'].rolling(LOCAL_WINDOW, min_periods=LOCAL_WINDOW).mean()
    trajectory['LPICP'] = lp
    trajectory['LE'] = ((1.0 - ALPHA) - lp).clip(lower=0)
    trajectory['local_overcoverage'] = (lp - (1.0 - ALPHA)).clip(lower=0)
    trajectory['local_abs_coverage_error'] = (lp - (1.0 - ALPHA)).abs()
    trajectory['local_MPIW'] = (trajectory['y_u_pred'] - trajectory['y_l_pred']).rolling(
        LOCAL_WINDOW, min_periods=LOCAL_WINDOW).mean()

    elapsed = time.monotonic() - t0
    metrics = summarize_all_metrics(
        trajectory['y_true'].to_numpy(),
        trajectory['y_l_pred'].to_numpy(),
        trajectory['y_u_pred'].to_numpy(),
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA, time_taken=elapsed,
    )
    metrics['dataset'] = name + '.csv'
    metrics['method'] = 'LACI_midpoint_absolute'
    metrics['time_sec'] = elapsed
    metrics['mean_radius'] = float(trajectory['radius'].mean())
    metrics['median_radius'] = float(trajectory['radius'].median())
    metrics['mean_alpha'] = float(trajectory['alpha_used'].mean())
    metrics['mean_weighted_err'] = float(trajectory['weighted_err'].mean())

    if SAVE_TRAJECTORIES:
        outdir = Path(TRAJECTORY_DIR)
        outdir.mkdir(parents=True, exist_ok=True)
        trajectory.to_csv(outdir / f'{name}__LACI_midpoint_absolute.csv', index=False)

    to_show = ['PICP', 'MPIW', 'LPICE', 'LPIA', 'LMPIW', 'time_sec']
    print(f'[{name}/LACI_midpoint_absolute] ' + ' | '.join(
        f'{k}={float(metrics[k]):.4f}' for k in to_show if k in metrics
    ), flush=True)
    return metrics


def self_test():
    # New nonconformity and interval inclusion are exactly equivalent.
    x = np.array([3.0, 5.0, 10.0])
    pred = np.array([4.0, 5.0, 7.0])
    assert np.allclose(absolute_score(x, pred), [1.0, 0.0, 3.0])
    for point in [0.0, 3.0]:
        for target in [-2.0, 0.0, 1.0, 5.0]:
            for rad in [0.0, 1.0, 4.0]:
                assert (point - rad <= target <= point + rad) == (abs(target-point) <= rad)
    assert weighted_laci_error([0, 1], [0, 0]) > 0.5
    assert np.isclose(weighted_laci_error([0, 1], [0, 0]), 1.0/(1.0+0.85))
    assert calibrated_radius(np.array([0.0, 1.0, 2.0, 3.0]), 0.1) >= 0
    print('Self-test passed: absolute score, symmetric interval, LACI weights, radius.')


def main(paths):
    print('Device:', DEVICE)
    all_metrics = []
    for path in paths:
        if not Path(path).is_file():
            raise FileNotFoundError(f'Missing dataset: {path}')
        name = Path(path).stem
        print(f'\nRunning {name} ...', flush=True)
        X, y, scaler = load_dataset(path)
        result = run_laci(X, y, scaler, name)
        all_metrics.append(result)
        pd.DataFrame(all_metrics).to_csv(OUTPUT_CSV, index=False)
    result_df = pd.DataFrame(all_metrics)
    keys = ['dataset', 'method', 'PICP', 'MPIW', 'MLPICP', 'LPICE',
            'LPIA', 'LMPIW', 'LCWC', 'STD_LE', 'n_local_windows', 'time_sec']
    keys = [k for k in keys if k in result_df.columns]
    result_df = result_df[keys + [k for k in result_df.columns if k not in keys]]
    result_df.to_csv(OUTPUT_CSV, index=False)
    print('\nResults saved:', OUTPUT_CSV)
    print(result_df.to_string(index=False))


if __name__ == '__main__':
    main(DATASET_PATHS)


Device: cuda

Running Plant_1_Generation_Data ...
[Plant_1_Generation_Data/LACI] 200/3331 PICP=0.5900 alpha=0.0240 radius=0.5984
[Plant_1_Generation_Data/LACI] 400/3331 PICP=0.7675 alpha=0.0177 radius=0.4796
[Plant_1_Generation_Data/LACI] 600/3331 PICP=0.8200 alpha=0.0173 radius=0.5919
[Plant_1_Generation_Data/LACI] 800/3331 PICP=0.8650 alpha=0.0373 radius=0.5130
[Plant_1_Generation_Data/LACI] 1000/3331 PICP=0.8920 alpha=0.0573 radius=0.5051
[Plant_1_Generation_Data/LACI] 1200/3331 PICP=0.9100 alpha=0.0773 radius=0.4713
[Plant_1_Generation_Data/LACI] 1400/3331 PICP=0.9229 alpha=0.0973 radius=0.4851
[Plant_1_Generation_Data/LACI] 1600/3331 PICP=0.9325 alpha=0.1173 radius=0.5204
[Plant_1_Generation_Data/LACI] 1800/3331 PICP=0.9400 alpha=0.1373 radius=0.1883
[Plant_1_Generation_Data/LACI] 2000/3331 PICP=0.9340 alpha=0.1172 radius=0.0467
[Plant_1_Generation_Data/LACI] 2200/3331 PICP=0.9250 alpha=0.0946 radius=0.0401
[Plant_1_Generation_Data/LACI] 2400/3331 PICP=0.9296 alpha=0.1106 radius=0